# Kisan Sahayak - Module 1: Crop Disease Detection

This notebook trains the model that looks at a leaf photo and names the disease.

**How to run it**

1. Open it in Google Colab.
2. Menu: **Runtime > Change runtime type > T4 GPU > Save**.
3. Menu: **Runtime > Run all**. The full run takes roughly 40 to 60 minutes.
4. The last cell downloads `kisan_disease_model.zip`. Unzip it into the project folder so that
   `models/` and `reports/` are filled.

**What the notebook does**

| Step | What happens |
| --- | --- |
| 1 | Check that the GPU is on |
| 2 | Download the PlantVillage dataset (about 54,000 leaf photos, 38 classes) |
| 3 | Look at the data: class counts and sample photos |
| 4 | Split into train / validation / test |
| 5 | Build the input pipeline and the augmentation |
| 6 | Build the model (MobileNetV2 + our own final layer) |
| 7 | Training step 1: train only the new final layer |
| 8 | Training step 2: fine-tune the last MobileNetV2 layers |
| 9 | Evaluate: accuracy, precision, recall, F1-score, confusion matrix |
| 10 | Save the model and download it |

## Settings

All the numbers you may want to change are in this one cell.

In [ ]:
IMAGE_SIZE = 224          # MobileNetV2 was designed for 224 x 224 pixel images
BATCH_SIZE = 32           # how many photos the model sees before each weight update
HEAD_EPOCHS = 8           # step 1: epochs for the new final layer
FINE_TUNE_EPOCHS = 8      # step 2: epochs with the last MobileNetV2 layers unfrozen
FINE_TUNE_LAYERS = 30     # how many of MobileNetV2's last layers to unfreeze
SEED = 42                 # fixed seed, so the split is the same on every run

DATA_DIR = "plantvillage"   # one sub-folder per class
MODEL_DIR = "models"
REPORTS_DIR = "reports"

## Step 1 - Check the GPU

Training on a GPU is about 20 times faster than on a CPU. If this prints an empty list,
turn the GPU on (Runtime > Change runtime type) and run the notebook again.

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(SEED)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Step 2 - Download the dataset

PlantVillage is a public dataset of leaf photos taken on a plain background. Each of its
38 folders is one class, named `Crop___Disease` (for example `Tomato___Late_blight`) or
`Crop___healthy`.

The GitHub repository holds three versions of every photo (colour, grayscale, segmented).
We only need the colour one, so the commands below download just that folder (about 600 MB).

In [ ]:
%%bash
if [ ! -d plantvillage ]; then
  git clone --quiet --filter=blob:none --no-checkout --depth 1 \
      https://github.com/spMohanty/PlantVillage-Dataset.git pv_repo
  cd pv_repo
  git sparse-checkout set --no-cone "/raw/color/"
  git checkout --quiet
  cd ..
  mv pv_repo/raw/color plantvillage
  rm -rf pv_repo
fi
echo "Classes: $(ls plantvillage | wc -l)"
echo "Images:  $(find plantvillage -type f | wc -l)"

## Step 3 - Look at the data

We collect the path of every image and its class number (0 to 37). The class number is
simply the position of the folder name in the alphabetically sorted list.

In [ ]:
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png")

class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))

paths, labels = [], []
for index, name in enumerate(class_names):
    folder = os.path.join(DATA_DIR, name)
    for file in sorted(os.listdir(folder)):
        if file.lower().endswith(IMAGE_SUFFIXES):
            paths.append(os.path.join(folder, file))
            labels.append(index)

labels = np.array(labels)
print(f"{len(paths)} images in {len(class_names)} classes")


def readable(class_name):
    """'Tomato___Late_blight' -> 'Tomato - Late blight'"""
    crop, _, condition = class_name.partition("___")
    return f"{crop.replace('_', ' ').strip()} - {condition.replace('_', ' ').strip()}"


readable_names = [readable(name) for name in class_names]

**Class balance.** The chart shows how many photos each class has. Some classes have more
than 5,000 photos and some fewer than 200. A model trained on such data tends to ignore the
small classes, which is why we use *class weights* later.

In [ ]:
counts = np.bincount(labels, minlength=len(class_names))

plt.figure(figsize=(10, 9))
plt.barh(readable_names, counts, color="seagreen")
plt.gca().invert_yaxis()
plt.xlabel("Number of photos")
plt.title("Photos per class")
plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/class_distribution.png", dpi=150)
plt.show()

print("Smallest class:", readable_names[counts.argmin()], counts.min())
print("Largest class: ", readable_names[counts.argmax()], counts.max())

In [ ]:
# Nine random photos with their class names
rng = np.random.default_rng(SEED)
plt.figure(figsize=(10, 10))
for i, index in enumerate(rng.choice(len(paths), 9, replace=False)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(plt.imread(paths[index]))
    plt.title(readable_names[labels[index]], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Step 4 - Split into train, validation and test

- **Train (70%)**: the photos the model learns from.
- **Validation (15%)**: checked after every epoch to see whether the model is still improving
  and to decide when to stop.
- **Test (15%)**: used only once, at the very end. These numbers go in the report, because the
  model has never seen these photos in any way.

`stratify` keeps the class mix the same in all three parts.

In [ ]:
train_paths, rest_paths, train_labels, rest_labels = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    rest_paths, rest_labels, test_size=0.50, stratify=rest_labels, random_state=SEED)

print(f"Train: {len(train_paths)}  Validation: {len(val_paths)}  Test: {len(test_paths)}")

## Step 5 - Input pipeline and augmentation

`tf.data` reads the photos from disk in the background, resizes them to 224 x 224 and groups
them into batches of 32, so the GPU never waits for data.

**Augmentation** shows the model a slightly different version of each photo in every epoch
(flipped, rotated, zoomed, different contrast). A farmer's photo will never be framed exactly
like the training photos, so this teaches the model to ignore those differences. It also
reduces overfitting. Augmentation is active only during training, never during prediction.

In [ ]:
def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image.set_shape((IMAGE_SIZE, IMAGE_SIZE, 3))
    return image, label          # pixel values stay 0-255; the model rescales them itself


def make_dataset(paths, labels, shuffle):
    dataset = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if shuffle:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_paths, train_labels, shuffle=True)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),      # up to +/- 54 degrees
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")

In [ ]:
# The same photo after augmentation, nine times
sample_images, _ = next(iter(train_ds))
plt.figure(figsize=(8, 8))
for i in range(9):
    augmented = augmentation(sample_images[:1], training=True)
    plt.subplot(3, 3, i + 1)
    plt.imshow(np.clip(augmented[0].numpy(), 0, 255).astype("uint8"))
    plt.axis("off")
plt.suptitle("One training photo, augmented nine different ways")
plt.tight_layout()
plt.show()

## Step 6 - Build the model (transfer learning)

Training a CNN from scratch needs millions of images. **Transfer learning** avoids that:

- **MobileNetV2** was already trained by Google on ImageNet (1.2 million photos, 1,000 classes).
  Its layers have learned to detect edges, colours, textures and shapes, which are just as
  useful for leaves.
- We remove its final 1,000-class layer (`include_top=False`) and put our own on top:
  - `GlobalAveragePooling2D` turns MobileNetV2's 7 x 7 x 1280 output into 1,280 numbers.
  - `Dropout(0.2)` randomly switches off 20% of them during training, to reduce overfitting.
  - `Dense(38, softmax)` gives one probability per class; the 38 probabilities add up to 1.

MobileNetV2 is small (about 3.5 million parameters, a 10 MB file) and fast, so the finished
app can run on a normal laptop or phone without a GPU.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False      # step 1: keep everything Google trained exactly as it is

inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="leaf_photo")
x = augmentation(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1.0)(x)    # 0..255 -> -1..1, the range MobileNetV2 expects
x = base_model(x, training=False)                  # training=False keeps BatchNorm statistics fixed
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax", name="disease")(x)

model = keras.Model(inputs, outputs, name="kisan_sahayak_mobilenetv2")
model.summary(show_trainable=True)

**Class weights.** A class with few photos gets a larger weight, so a mistake on it costs the
model more. Without this the model could reach a high accuracy just by being good at the big
classes.

In [ ]:
weights = compute_class_weight("balanced", classes=np.arange(len(class_names)), y=train_labels)
class_weight = dict(enumerate(weights))

MODEL_PATH = f"{MODEL_DIR}/plant_disease.keras"

# Saves the model only when validation accuracy improves, so the file always holds the best epoch.
checkpoint = keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True)


def early_stopping():
    # Stops training when validation accuracy has not improved for 3 epochs.
    return keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3)

## Step 7 - Training step 1: the new final layer

Only the `Dense` layer is trained here (about 49,000 parameters). The loss function is
*sparse categorical cross-entropy*, the standard choice when each photo belongs to exactly
one class and the labels are class numbers.

In [ ]:
started = time.time()

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_head = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
                         class_weight=class_weight, callbacks=[early_stopping(), checkpoint])

## Step 8 - Training step 2: fine-tuning

Now we unfreeze the last 30 layers of MobileNetV2 so they can adapt from "general photos" to
"leaves". The learning rate is 100 times smaller (0.00001): large updates would destroy what
those layers already know.

The early layers stay frozen. They detect basic edges and colours, which do not need to change.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
                         class_weight=class_weight, callbacks=[early_stopping(), checkpoint])

training_minutes = round((time.time() - started) / 60, 1)
print(f"Training took {training_minutes} minutes")

**Training curves.** Training and validation lines that stay close together mean the model
generalises well. If training accuracy keeps rising while validation accuracy falls, the model
is overfitting (memorising the training photos).

In [ ]:
history = {key: [float(v) for v in history_head.history[key] + history_fine.history[key]]
           for key in history_head.history}
fine_tune_start = len(history_head.history["loss"])
epochs = range(1, len(history["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric, title in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
    axis.plot(epochs, history[metric], marker="o", label="Training")
    axis.plot(epochs, history[f"val_{metric}"], marker="o", label="Validation")
    axis.axvline(fine_tune_start + 0.5, color="grey", linestyle="--", label="Fine-tuning starts")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.legend()
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/training_curves.png", dpi=150)
plt.show()

## Step 9 - Evaluate on the test set

We load the best saved epoch and predict the 15% of photos the model has never seen.

- **Accuracy**: of all test photos, how many were classified correctly.
- **Precision** (per class): when the model says "late blight", how often it is right.
- **Recall** (per class): of all real late-blight photos, how many the model found.
- **F1-score**: one number combining precision and recall (their harmonic mean).
- **Macro average**: the plain average over the 38 classes, so small classes count as much
  as big ones. **Weighted average** counts each class by its number of photos.

In [ ]:
model = keras.models.load_model(MODEL_PATH)       # the best epoch, not necessarily the last

probabilities = model.predict(test_ds)
predicted = probabilities.argmax(axis=1)

accuracy = accuracy_score(test_labels, predicted)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, predicted, average="macro", zero_division=0)

print(f"Test accuracy     : {accuracy:.4f}")
print(f"Macro precision   : {precision:.4f}")
print(f"Macro recall      : {recall:.4f}")
print(f"Macro F1-score    : {f1:.4f}")
print()

report = classification_report(test_labels, predicted, labels=range(len(class_names)),
                               target_names=readable_names, digits=3, zero_division=0)
print(report)
with open(f"{REPORTS_DIR}/classification_report.txt", "w", encoding="utf-8") as file:
    file.write(report)

**Confusion matrix.** Each row is the real class and each column is what the model predicted.
Numbers on the diagonal are correct predictions. A number off the diagonal shows two classes
the model mixes up; expect most of those among diseases of the same crop that look alike.

In [ ]:
matrix = confusion_matrix(test_labels, predicted, labels=range(len(class_names)))

plt.figure(figsize=(18, 16))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Greens", cbar=False,
            xticklabels=readable_names, yticklabels=readable_names, annot_kws={"size": 7})
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix on the test set")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/confusion_matrix.png", dpi=150)
plt.show()

# The five pairs the model confuses most often
errors = matrix.copy()
np.fill_diagonal(errors, 0)
print("Most common mistakes (actual -> predicted):")
for flat_index in errors.flatten().argsort()[::-1][:5]:
    actual, wrong = divmod(flat_index, len(class_names))
    if errors[actual, wrong]:
        print(f"  {errors[actual, wrong]:3d} x  {readable_names[actual]}  ->  {readable_names[wrong]}")

**Confidence check.** The app warns the farmer when the model's confidence is below 70% and
advises visiting the Krishi Vigyan Kendra. This cell shows how often that warning would appear
on the test photos, and how accurate the model is above and below the line.

In [ ]:
THRESHOLD = 0.70
confidence = probabilities.max(axis=1)
correct = predicted == test_labels
confident = confidence >= THRESHOLD

print(f"Predictions with confidence >= 70%: {confident.mean():.1%} of test photos, "
      f"accuracy {correct[confident].mean():.1%}")
if (~confident).any():
    print(f"Predictions with confidence <  70%: {(~confident).mean():.1%} of test photos, "
          f"accuracy {correct[~confident].mean():.1%}")

## Step 10 - Save and download

Three things are saved:

- `models/plant_disease.keras`: the trained model.
- `models/class_names.json`: the class list in the order the model outputs it. Without it the
  app could not turn "class 29" back into "Tomato - Early blight".
- `reports/metrics.json`: every number above, so the app and the project report use the real
  results.

In [ ]:
with open(f"{MODEL_DIR}/class_names.json", "w", encoding="utf-8") as file:
    json.dump({"image_size": IMAGE_SIZE, "class_names": class_names}, file, indent=2)

metrics = {
    "test_accuracy": float(accuracy),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(f1),
    "best_val_accuracy": max(history["val_accuracy"]),
    "images": {"total": len(paths), "train": len(train_paths),
               "validation": len(val_paths), "test": len(test_paths)},
    "classes": len(class_names),
    "image_size": IMAGE_SIZE,
    "epochs": len(history["loss"]),
    "fine_tune_epochs": len(history_fine.history["loss"]),
    "training_minutes": training_minutes,
    "history": history,
}
with open(f"{REPORTS_DIR}/metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print(json.dumps({k: v for k, v in metrics.items() if k != "history"}, indent=2))

In [ ]:
# Zip the model and the reports and download them to your computer
!zip -q -r kisan_disease_model.zip models reports

try:
    from google.colab import files
    files.download("kisan_disease_model.zip")
except ImportError:
    print("Not running in Colab: kisan_disease_model.zip is in the current folder.")

## What to do next

1. Unzip `kisan_disease_model.zip` into the project folder (replace the existing `models/` and
   `reports/` files).
2. Copy the printed numbers from Step 9 and Step 10 back to your mentor chat. Only those real
   numbers go into the report and the presentation.